# NB03 — Baseline + Eksternal, 4 Dataset

Hibah Kemdiktisaintek Part 3. 5 arsitektur eksternal (Abdelli-BiGRU, Soothar-CNNLSTM,
Yang-IMCBiLSTM, Healy-GRUClassifier, Titouni-CNNExtractor) + 5 baseline konvensional
(Baseline-CNNOnly, Baseline-BiLSTMOnly, Baseline-LogReg, Baseline-RandomForest,
Baseline-SVM), di KEEMPAT dataset (bukan cuma OTDR+SOR seperti draf Part 2 lama),
protokol final IDENTIK dengan HDMC (10-fold/1-repeat), WAJIB supaya uji Wilcoxon
berpasangan (nb-uji signifikansi) valid.

Model TIAP FOLD tersimpan penuh (perbaikan dari Part 2 yang cuma simpan 1 model final).

Estimasi waktu (dari perhitungan awal Part 3, skala dari data timing lama):
OTDR ~44 menit, SOR ~5,3 jam (didominasi SVM linear di ~100rb baris),
HardFailure+SoftFailure gabungan ~35-40 menit. Total ~6-7 jam, bisa ditinggal jalan.

In [ ]:
import os, sys, time
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import DATASET_ORDER, N_FOLDS_FINAL, N_REPEATS_FINAL, TAB_DIR
from src.comparator_runner import run_all_comparators_one_dataset

print("Urutan dataset:", DATASET_ORDER)
print(f"Protokol final: n_folds={N_FOLDS_FINAL}, n_repeats={N_REPEATS_FINAL} (SAMA dengan HDMC di nb02)")

In [ ]:
### Redirect output ke results_clean/ (tidak menimpa results/ asli)
import os
RESULTS_CLEAN_ROOT = os.path.abspath(os.path.join("..", "results_clean"))
for _sub in ["tables", "figures", "models", "predictions"]:
    os.makedirs(os.path.join(RESULTS_CLEAN_ROOT, _sub), exist_ok=True)

_new_tab_dir = os.path.join(RESULTS_CLEAN_ROOT, "tables")
_new_mdl_dir = os.path.join(RESULTS_CLEAN_ROOT, "models")
_new_pred_dir = os.path.join(RESULTS_CLEAN_ROOT, "predictions")
_new_fig_dir = os.path.join(RESULTS_CLEAN_ROOT, "figures")

import src.hdmc_multiview as _m1
import src.comparator_runner as _m2
import src.experiment_runner as _m3
import src.justification_runner as _m4
import src.data_quality as _m5

for _mod in [_m1, _m2, _m3, _m4, _m5]:
    if hasattr(_mod, "TAB_DIR"):
        _mod.TAB_DIR = _new_tab_dir
    if hasattr(_mod, "MDL_DIR"):
        _mod.MDL_DIR = _new_mdl_dir
    if hasattr(_mod, "PRED_DIR"):
        _mod.PRED_DIR = _new_pred_dir

TAB_DIR = _new_tab_dir
MDL_DIR = _new_mdl_dir
PRED_DIR = _new_pred_dir
FIG_DIR = _new_fig_dir

print(f"Output diarahkan ke: {RESULTS_CLEAN_ROOT}")

## Jalankan 10 pembanding di keempat dataset

Urutan SOR dulu (paling berat, SVM ~5 jam sendirian di sini), supaya kalau ada
masalah di tengah jalan, ketahuan lebih awal daripada di dataset terakhir.

In [ ]:
comparator_results = {}
t0_total = time.perf_counter()
for ds in DATASET_ORDER:
    t0 = time.perf_counter()
    hasil = run_all_comparators_one_dataset(ds, verbose=True)
    elapsed = time.perf_counter() - t0
    comparator_results[ds] = hasil
    print(f"\n[{ds}] SEMUA 10 PEMBANDING SELESAI dalam {elapsed/60:.1f} menit "
          f"({elapsed/3600:.2f} jam)")

total_elapsed = time.perf_counter() - t0_total
print(f"\n{'='*70}\nSELURUH NB03 SELESAI dalam {total_elapsed/60:.1f} menit "
      f"({total_elapsed/3600:.2f} jam)\n{'='*70}")

In [ ]:
comparator_results = {}

import importlib, os, json
import src.comparator_runner, src.comparator_architectures
importlib.reload(src.comparator_architectures)
importlib.reload(src.comparator_runner)
from src.comparator_runner import run_classical_baseline, run_deep_model, EXTERNAL_NAMES, DEEP_BASELINE_NAMES, CLASSICAL_BASELINES
from src.comparator_architectures import COMPARATOR_REGISTRY
from src.metrics import aggregate_fold_metrics
from src.config import MDL_DIR, N_FOLDS_FINAL, TAB_DIR
from src.hdmc_multiview import load_dataset

def rekonstruksi_dari_disk(dataset_name, model_name, class_names, n_folds=N_FOLDS_FINAL):
    model_dir = os.path.join(MDL_DIR, "comparators", dataset_name, model_name)
    fold_metrics = []
    for i in range(n_folds):
        with open(os.path.join(model_dir, f"fold_{i:02d}_metrics.json")) as f:
            fold_metrics.append(json.load(f))
    agg = aggregate_fold_metrics(fold_metrics)
    agg["dataset"] = dataset_name
    agg["model_name"] = model_name
    agg["class_names"] = class_names
    agg["accuracy_per_fold"] = [m["accuracy"] for m in fold_metrics]
    agg["macro_f1_per_fold"] = [m["macro_f1"] for m in fold_metrics]
    agg["bacc_per_fold"] = [m["bacc"] for m in fold_metrics]
    agg["mcc_per_fold"] = [m["mcc"] for m in fold_metrics]
    return agg

X, y, class_names, groups = load_dataset("sor")

# 9 model yang SUDAH selesai, rekonstruksi dari disk (tidak training ulang)
hasil_sor = {}
for name in EXTERNAL_NAMES + ["Titouni-CNNExtractor"] + DEEP_BASELINE_NAMES + ["Baseline-LogReg", "Baseline-RandomForest"]:
    hasil_sor[name] = rekonstruksi_dari_disk("sor", name, class_names)
    print(f"  [rekonstruksi] {name}: macro_f1={hasil_sor[name]['macro_f1_mean']:.4f}")

# SVM: training BARU (sudah di-fix, cepat)
hasil_sor["Baseline-SVM"] = run_classical_baseline(
    "sor", "Baseline-SVM", X, y, class_names, groups=groups, verbose=True,
)

# BiGRU: TAMBAHAN BARU khusus SOR (model ke-11)
hasil_sor["Baseline-BiGRUOnly"] = run_deep_model(
    "sor", "Baseline-BiGRUOnly", COMPARATOR_REGISTRY, X, y, class_names,
    groups=groups, verbose=True,
)

out_path = os.path.join(TAB_DIR, "comparator_sor.json")
with open(out_path, "w") as f:
    json.dump(hasil_sor, f, indent=2, default=str)
print(f"\nTersimpan: {out_path}")

comparator_results["sor"] = hasil_sor
print("comparator_results['sor'] terisi,", len(hasil_sor), "model")

In [ ]:
if "comparator_results" not in locals(): comparator_results = {}

from src.hdmc_multiview import load_dataset
import os, json
X, y, class_names, groups = load_dataset("sor")
hasil_sor = {}
def rekonstruksi_dari_disk(dataset_name, model_name, class_names, n_folds=N_FOLDS_FINAL):
    from src.metrics import aggregate_fold_metrics
    from src.config import MDL_DIR
    model_dir = os.path.join(MDL_DIR, "comparators", dataset_name, model_name)
    fold_metrics = []
    for i in range(n_folds):
        with open(os.path.join(model_dir, f"fold_{i:02d}_metrics.json")) as f:
            fold_metrics.append(json.load(f))
    agg = aggregate_fold_metrics(fold_metrics)
    agg["dataset"] = dataset_name
    agg["model_name"] = model_name
    agg["class_names"] = class_names
    agg["accuracy_per_fold"] = [m["accuracy"] for m in fold_metrics]
    agg["macro_f1_per_fold"] = [m["macro_f1"] for m in fold_metrics]
    agg["bacc_per_fold"] = [m["bacc"] for m in fold_metrics]
    agg["mcc_per_fold"] = [m["mcc"] for m in fold_metrics]
    return agg

from src.comparator_runner import DEEP_BASELINE_NAMES, CLASSICAL_BASELINES
print("Merekonstruksi model baseline yang tidak di-train ulang...")
for name in DEEP_BASELINE_NAMES + list(CLASSICAL_BASELINES.keys()):
    try:
        hasil_sor[name] = rekonstruksi_dari_disk("sor", name, class_names)
    except Exception as e:
        print(f"Skip {name}, belum ada di disk.")

# Latih Ulang 5 Model Eksternal SAJA untuk dataset SOR
# (menimpa hasil lama yang belum menggunakan fitur perbaikan)
print("\n--- MELATIH ULANG 5 MODEL EKSTERNAL UNTUK SOR ---")
import src.comparator_runner, src.comparator_architectures
import importlib
importlib.reload(src.comparator_architectures)
importlib.reload(src.comparator_runner)
from src.comparator_runner import EXTERNAL_NAMES, run_deep_model, run_titouni_ensemble
from src.comparator_architectures import EXTERNAL_REGISTRY

external_configs = {
    "Abdelli-BiGRU": {"loss_type": "CE", "resampler": "None"},
    "Soothar-CNNLSTM": {"loss_type": "CE", "resampler": "None"},
    "Yang-IMCBiLSTM": {"loss_type": "CE", "resampler": "None"},
    "Healy-GRUClassifier": {"loss_type": "CE", "resampler": "cVAE"}
}
for name in EXTERNAL_NAMES:
    config = external_configs[name]
    hasil_sor[name] = run_deep_model(
        "sor", name, EXTERNAL_REGISTRY, X, y, class_names,
        groups=groups, verbose=True,
        loss_type=config["loss_type"], resampler=config["resampler"]
    )

# Titouni Meta Ensemble dipisah karena pipeline khusus
hasil_sor["Titouni-CNNExtractor"] = run_titouni_ensemble(
    "sor", "Titouni-CNNExtractor", EXTERNAL_REGISTRY["Titouni-CNNEnsemble"],
    X, y, class_names, groups=groups, verbose=True
)

out_path = os.path.join(TAB_DIR, "comparator_sor.json")
with open(out_path, "w") as f:
    json.dump(hasil_sor, f, indent=2, default=str)
print(f"\nTersimpan ulang: {out_path}")

comparator_results["sor"] = hasil_sor


In [ ]:
if "comparator_results" not in locals(): comparator_results = {}

from src.hdmc_multiview import load_dataset
import os, json
X, y, class_names, groups = load_dataset("sor")
hasil_sor = {}
def rekonstruksi_dari_disk(dataset_name, model_name, class_names, n_folds=N_FOLDS_FINAL):
    from src.metrics import aggregate_fold_metrics
    from src.config import MDL_DIR
    model_dir = os.path.join(MDL_DIR, "comparators", dataset_name, model_name)
    fold_metrics = []
    for i in range(n_folds):
        with open(os.path.join(model_dir, f"fold_{i:02d}_metrics.json")) as f:
            fold_metrics.append(json.load(f))
    agg = aggregate_fold_metrics(fold_metrics)
    agg["dataset"] = dataset_name
    agg["model_name"] = model_name
    agg["class_names"] = class_names
    agg["accuracy_per_fold"] = [m["accuracy"] for m in fold_metrics]
    agg["macro_f1_per_fold"] = [m["macro_f1"] for m in fold_metrics]
    agg["bacc_per_fold"] = [m["bacc"] for m in fold_metrics]
    agg["mcc_per_fold"] = [m["mcc"] for m in fold_metrics]
    return agg

from src.comparator_runner import DEEP_BASELINE_NAMES, CLASSICAL_BASELINES
print("Merekonstruksi model baseline yang tidak di-train ulang...")
for name in DEEP_BASELINE_NAMES + list(CLASSICAL_BASELINES.keys()):
    try:
        hasil_sor[name] = rekonstruksi_dari_disk("sor", name, class_names)
    except Exception as e:
        print(f"Skip {name}, belum ada di disk.")

# Latih Ulang 5 Model Eksternal SAJA untuk dataset SOR
# (menimpa hasil lama yang belum menggunakan fitur perbaikan)
print("\n--- MELATIH ULANG 5 MODEL EKSTERNAL UNTUK SOR ---")
import src.comparator_runner, src.comparator_architectures
import importlib
importlib.reload(src.comparator_architectures)
importlib.reload(src.comparator_runner)
from src.comparator_runner import EXTERNAL_NAMES, run_deep_model, run_titouni_ensemble
from src.comparator_architectures import EXTERNAL_REGISTRY


In [ ]:
# ============================================================
# Lanjutkan Healy dan Titouni SAJA (3 model lain sudah selesai)
# ============================================================
import importlib, os, json
import src.comparator_runner, src.comparator_architectures, src.resampling_and_loss
importlib.reload(src.comparator_architectures)
importlib.reload(src.resampling_and_loss)
importlib.reload(src.comparator_runner)
from src.comparator_runner import run_deep_model, run_titouni_ensemble
from src.comparator_architectures import EXTERNAL_REGISTRY
from src.config import TAB_DIR

# Pastikan variabel tersedia (jalankan setelah load_dataset dari cell sebelumnya)
# X, y, class_names, groups, hasil_sor, comparator_results harus sudah ada

print("Training Healy-GRUClassifier dengan cVAE...")
hasil_sor["Healy-GRUClassifier"] = run_deep_model(
    "sor", "Healy-GRUClassifier", EXTERNAL_REGISTRY, X, y, class_names,
    groups=groups, verbose=True,
    loss_type="CE", resampler="cVAE"
)

print("\nTraining Titouni Meta-Ensemble...")
hasil_sor["Titouni-CNNExtractor"] = run_titouni_ensemble(
    "sor", "Titouni-CNNExtractor", EXTERNAL_REGISTRY["Titouni-CNNEnsemble"],
    X, y, class_names, groups=groups, verbose=True
)

out_path = os.path.join(TAB_DIR, "comparator_sor.json")
with open(out_path, "w") as f:
    json.dump(hasil_sor, f, indent=2, default=str)
print(f"\nTersimpan ulang: {out_path}")

comparator_results["sor"] = hasil_sor
print("comparator_results['sor'] diperbarui,", len(hasil_sor), "model")


In [ ]:
for ds in ["otdr", "hardfailure", "softfailure"]:
    t0 = time.perf_counter()
    hasil = run_all_comparators_one_dataset(
        ds, verbose=True,
        deep_baseline_names=["Baseline-CNNOnly", "Baseline-BiGRUOnly"],  # BiLSTM -> BiGRU
    )
    comparator_results[ds] = hasil
    print(f"\n[{ds}] SELESAI dalam {(time.perf_counter()-t0)/60:.1f} menit")

### Ringkasan hasil 10 pembanding x 4 dataset

In [ ]:
import pandas as pd
rows = []
for ds, models in comparator_results.items():
    for model_name, agg in models.items():
        rows.append({
            "dataset": ds, "model": model_name,
            "macro_f1_mean": round(agg["macro_f1_mean"], 4),
            "macro_f1_std": round(agg["macro_f1_std"], 4),
            "bacc_mean": round(agg["bacc_mean"], 4),
            "mcc_mean": round(agg["mcc_mean"], 4),
            "n_observations": agg["n_observations"],
        })
df_comparator_summary = pd.DataFrame(rows)
df_comparator_summary = df_comparator_summary.sort_values(["dataset", "macro_f1_mean"], ascending=[True, False])
print(df_comparator_summary.to_string(index=False))

out_path = os.path.join(TAB_DIR, "comparator_ringkasan_4_dataset.csv")
df_comparator_summary.to_csv(out_path, index=False)
print(f"\nTersimpan: {out_path}")

### Bandingkan cepat terhadap HDMC (dari nb02)

Sanity check kasar sebelum uji signifikansi formal di nb-uji signifikansi. Ini BUKAN
pengganti uji statistik, cuma pengecekan cepat urutan performa masuk akal.

In [ ]:
import json as _json
hdmc_path = os.path.join(TAB_DIR, "hdmc_ringkasan_4_dataset.csv")
if os.path.exists(hdmc_path):
    df_hdmc = pd.read_csv(hdmc_path)
    print("HDMC (dari nb02):")
    print(df_hdmc[["dataset", "macro_f1_mean"]].to_string(index=False))
    print("\nTerbaik per dataset dari 10 pembanding (nb03):")
    best_per_ds = df_comparator_summary.loc[df_comparator_summary.groupby("dataset")["macro_f1_mean"].idxmax()]
    print(best_per_ds[["dataset", "model", "macro_f1_mean"]].to_string(index=False))
else:
    print("hdmc_ringkasan_4_dataset.csv belum ada, jalankan nb02 dulu untuk perbandingan ini.")

---
## Uji signifikansi: HDMC (proposed) vs seluruh 10 pembanding

Wilcoxon signed-rank berpasangan + Cliff's delta, HDMC sebagai baseline pembanding
(`baseline="HDMC"`), dihitung LENGKAP untuk keempat dataset x 10 pembanding (40
pasangan total). Subset mana yang masuk badan naskah jurnal (kemungkinan cuma
Titouni sebagai pembanding utama, sesuai instruksi proyek) diputuskan NANTI saat
penulisan, bukan di sini. Fungsi ini menghitung semuanya supaya pilihan itu
punya dasar data lengkap, bukan tebakan.

CATATAN VALIDITAS: Wilcoxon berpasangan mensyaratkan observasi HDMC dan pembanding
berasal dari fold yang SAMA PERSIS. Karena nb02 dan nb03 sama-sama pakai
get_repeated_folds()/get_repeated_folds_grouped() dengan seed dan n_folds/n_repeats
yang identik, fold-nya seharusnya sama, tapi urutan generator scikit-learn TETAP
bisa berbeda kalau X berbeda shape antar model (meta-feature HDMC vs fitur mentah
komparator). Baris "n_observations cocok?" di bawah WAJIB dicek sebelum
menyimpulkan p-value, bukan diasumsikan otomatis benar.

In [ ]:
# --- Setup: pastikan src/ bisa diimport & variabel tersedia (aman dijalankan ulang) ---
import os, sys
import json as _json
import pandas as pd

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import DATASET_ORDER, TAB_DIR
from src.significance_tests import wilcoxon_pairwise

# Rekonstruksi hdmc_results dari file JSON yang sudah tersimpan (hasil nb02b)
if 'hdmc_results' not in vars() or not hdmc_results:
    hdmc_results = {}
    for ds in DATASET_ORDER:
        json_path = os.path.join(TAB_DIR, f"{ds}_hdmc_final_metrics.json")
        if not os.path.exists(json_path):
            raise FileNotFoundError(f"File HDMC tidak ditemukan: {json_path}. Jalankan nb02b dulu.")
        with open(json_path) as f:
            hdmc_results[ds] = _json.load(f)
    print("hdmc_results dimuat dari file JSON:", list(hdmc_results.keys()))
else:
    print("hdmc_results sudah ada di memori kernel.")

# Rekonstruksi comparator_results dari file JSON yang sudah tersimpan (hasil nb03)
if 'comparator_results' not in vars() or not comparator_results:
    comparator_results = {}
    for ds in DATASET_ORDER:
        json_path = os.path.join(TAB_DIR, f"comparator_{ds}.json")
        if not os.path.exists(json_path):
            raise FileNotFoundError(f"File komparator tidak ditemukan: {json_path}. Jalankan cell training nb03 dulu.")
        with open(json_path) as f:
            comparator_results[ds] = _json.load(f)
    print("comparator_results dimuat dari file JSON:", list(comparator_results.keys()))
else:
    print("comparator_results sudah ada di memori kernel.")
# --- Akhir setup ---

sig_rows = []
for ds in DATASET_ORDER:
    scores = {"HDMC": hdmc_results[ds]["macro_f1_per_fold"]}
    for model_name, agg in comparator_results[ds].items():
        scores[model_name] = agg["macro_f1_per_fold"]

    n_obs = {k: len(v) for k, v in scores.items()}
    if len(set(n_obs.values())) > 1:
        print(f"[{ds}] [PERINGATAN] n_observations TIDAK SAMA antar kandidat: {n_obs}. "
              f"Wilcoxon di bawah untuk dataset ini TIDAK VALID sampai ini diperbaiki.")
        continue

    df_wil = wilcoxon_pairwise(scores, baseline="HDMC")
    df_wil.insert(0, "dataset", ds)
    sig_rows.append(df_wil)
    print(f"[{ds}] uji signifikansi HDMC vs {len(scores)-1} pembanding selesai "
          f"(n_observations={n_obs['HDMC']} tiap kandidat)")

df_signifikansi = pd.concat(sig_rows, ignore_index=True) if sig_rows else pd.DataFrame()
out_path = os.path.join(TAB_DIR, "uji_signifikansi_hdmc_vs_semua_pembanding.csv")
df_signifikansi.to_csv(out_path, index=False)
print(f"\nTersimpan: {out_path}")
print(df_signifikansi.to_string(index=False))

In [ ]:
# Ringkasan cepat: berapa pembanding yang p<0.05 (signifikan) per dataset
if not df_signifikansi.empty:
    ringkas = (df_signifikansi.assign(signifikan=df_signifikansi["p_value"] < 0.05)
               .groupby("dataset")["signifikan"].agg(["sum", "count"]))
    ringkas.columns = ["jumlah_signifikan", "jumlah_pembanding"]
    print(ringkas)
    print("\nCatatan: p>=0.05 pada pasangan tertentu (kemungkinan performa mirip HDMC) "
          "adalah sinyal untuk MEMPERTIMBANGKAN apakah pembanding itu tetap perlu ditulis "
          "di naskah, sesuai diskusi sebelumnya soal penelitian dengan hasil serupa "
          "berpotensi dihapus dari badan paper (bukan otomatis dihapus, tetap keputusan manual).")